In [13]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 02 — Limpeza e Padronização
# ============================================================
# Objetivo: Aplicar as decisões de limpeza definidas a partir
# do diagnóstico do Notebook 01, gerando um dataset confiável
# para as próximas etapas do pipeline.
#
# Decisões aplicadas:
# - Reviews vazios → REMOVIDOS
# - Reviews curtos (< 3 palavras) → MANTIDOS com flag
# - Emojis → REMOVIDOS
# - Datas PT-BR → convertidas para datetime
# - Texto → lowercase + remoção de caracteres especiais
# ============================================================

import pandas as pd
import json
import re
import os
from pathlib import Path

# Caminhos do projeto
ROOT = Path().resolve().parent
RAW_PATH = ROOT / "data" / "raw"
PROCESSED_PATH = ROOT / "data" / "processed"

ARQUIVO_1 = RAW_PATH / "reviews_mercadolivre_com_br_1.json"
ARQUIVO_2 = RAW_PATH / "reviews_mercadolivre_com_br_2.json"

# Carrega e unifica os dois JSONs
df1 = pd.read_json(ARQUIVO_1, encoding="utf-8")
df2 = pd.read_json(ARQUIVO_2, encoding="utf-8")
df = pd.concat([df1, df2], ignore_index=True)

# Adiciona coluna de quantidade de palavras
df["qtd_palavras"] = df["content"].fillna("").apply(lambda x: len(str(x).split()))

print(f"✅ Dataset carregado: {df.shape[0]:,} registros")
print(f"📋 Colunas: {list(df.columns)}")

✅ Dataset carregado: 206,952 registros
📋 Colunas: ['date', 'rating', 'content', 'product_url', 'qtd_palavras']


In [14]:
# ============================================================
# Remove registros onde 'content' é nulo ou string vazia
# Mantém reviews curtos, mas marca com flag
# ============================================================

total_antes = len(df)

# Remove nulos
df = df[df["content"].notna()]

# Remove strings vazias ou só espaços
df = df[df["content"].str.strip() != ""]

total_depois = len(df)
removidos = total_antes - total_depois

print(f"📊 Registros antes:   {total_antes:,}")
print(f"📊 Registros depois:  {total_depois:,}")
print(f"🗑️ Reviews vazios removidos: {removidos:,}")

📊 Registros antes:   206,952
📊 Registros depois:  202,785
🗑️ Reviews vazios removidos: 4,167


In [15]:
# ============================================================
# Cria coluna 'review_curto' para reviews com menos de 3 palavras
# Esses registros são mantidos no dataset mas sinalizados
# para que possam ser filtrados no Power BI quando necessário
# ============================================================

df["review_curto"] = df["qtd_palavras"] < 3

total_curtos = df["review_curto"].sum()
pct_curtos = (total_curtos / len(df) * 100).round(2)

print(f"🏷️Reviews marcados como curtos: {total_curtos:,} ({pct_curtos}% do total)")
print(f"\nExemplos de reviews curtos:")
display(df[df["review_curto"] == True][["rating", "content"]].head(8))

🏷️Reviews marcados como curtos: 54,545 (26.9% do total)

Exemplos de reviews curtos:


,rating,content
0,5,Top.
2,5,Ótima qualidade.
3,4,Bom.
7,5,Amei 😍.
10,5,Amei.
12,5,Muito bom!.
14,5,Muito bom.
18,5,Amo.


In [16]:
# ============================================================
# Converte datas no formato PT-BR do Mercado Livre
# Exemplos: "19 ago. 2024", "09 set. 2023", "15 fev. 2025"
# ============================================================

# Dicionário de meses em português
MESES_PT = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12"
}

def converter_data(data_str):
    """
    Converte string de data PT-BR para datetime.
    Retorna NaT se não conseguir converter.
    """
    try:
        # Remove pontos, lowercase e divide
        partes = str(data_str).lower().replace(".", "").split()
        dia  = partes[0].zfill(2)
        mes  = MESES_PT.get(partes[1][:3], "00")
        ano  = partes[2]
        return pd.to_datetime(f"{ano}-{mes}-{dia}", format="%Y-%m-%d")
    except:
        return pd.NaT

df["date"] = df["date"].apply(converter_data)

datas_invalidas = df["date"].isna().sum()
print(f"✅ Datas convertidas para datetime")
print(f"⚠️ Datas que não puderam ser convertidas: {datas_invalidas:,}")
print(f"\nRange temporal:")
print(f"Mais antiga:  {df['date'].min().date()}")
print(f"Mais nova:     {df['date'].max().date()}")

df.head(10)


✅ Datas convertidas para datetime
⚠️ Datas que não puderam ser convertidas: 0

Range temporal:
Mais antiga:  2015-11-01
Mais nova:     2025-02-19


,date,rating,content,product_url,qtd_palavras,review_curto
0,2023-09-09,5,Top.,https://produto.mercadolivre.com.br/MLB-314957...,1,True
1,2024-08-19,5,"Produto bom, cumpre o que promete.",https://produto.mercadolivre.com.br/MLB-314957...,6,False
2,2025-02-15,5,Ótima qualidade.,https://produto.mercadolivre.com.br/MLB-314957...,2,True
3,2025-02-11,4,Bom.,https://produto.mercadolivre.com.br/MLB-314957...,1,True
4,2025-01-10,5,Atendeu minhas expectativas.,https://produto.mercadolivre.com.br/MLB-314957...,3,False
5,2025-02-18,5,Ótimo recomendo e já usei a fragrância é bem f...,https://produto.mercadolivre.com.br/MLB-314885...,21,False
6,2023-11-26,4,"Dentro do esperado, recomendo.",https://produto.mercadolivre.com.br/MLB-314885...,4,False
7,2023-09-16,5,Amei 😍.,https://produto.mercadolivre.com.br/MLB-314885...,2,True
8,2023-08-13,5,Se fizer direitinho o cabelo fica muito bom de...,https://produto.mercadolivre.com.br/MLB-314885...,40,False
9,2023-05-31,5,Excelente produto da um resultado perfeito e m...,https://produto.mercadolivre.com.br/MLB-314885...,16,False


In [17]:
# ============================================================
# Limpa o campo 'content':
# 1. Remove emojis 
# 2. Converte para lowercase
# 3. Remove caracteres especiais mantendo pontuação básica
# 4. Remove espaços extras
# ============================================================

def remover_emojis(texto):
    """
    Remove emojis e símbolos especiais usando range Unicode.
    """
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # símbolos e pictogramas
        u"\U0001F680-\U0001F6FF"  # transporte e mapas
        u"\U0001F1E0-\U0001F1FF"  # bandeiras
        u"\U00002500-\U00002BEF"  # símbolos chineses
        u"\U00002702-\U000027B0"  # dingbats
        u"\U000024C2-\U0001F251"  # enclosed characters
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub("", texto)

def limpar_texto(texto):
    """
    Pipeline completo de limpeza textual:
    1. Remove emojis
    2. Lowercase
    3. Remove caracteres especiais (mantém letras, números, espaços e pontuação básica)
    4. Remove espaços extras
    """
    if pd.isna(texto):
        return ""

    # 1. Remove emojis
    texto = remover_emojis(str(texto))

    # 2. Lowercase
    texto = texto.lower()

    # 3. Remove caracteres especiais — mantém acentos PT-BR
    texto = re.sub(r"[^a-záàãâéêíóôõúüçñ0-9\s\.\,\!\?]", " ", texto)

    # 4. Remove espaços extras
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

# Preserva o texto original em nova coluna antes de limpar
df["content_original"] = df["content"]

# Aplica a limpeza
df["content"] = df["content"].apply(limpar_texto)

# Recalcula qtd_palavras após limpeza
df["qtd_palavras"] = df["content"].apply(lambda x: len(str(x).split()))

print("✅ Limpeza textual aplicada")
print("\nComparação antes x depois:")
comparacao = df[["content_original", "content"]].head(8)
display(comparacao)

✅ Limpeza textual aplicada

Comparação antes x depois:


,content_original,content
0,Top.,top.
1,"Produto bom, cumpre o que promete.","produto bom, cumpre o que promete."
2,Ótima qualidade.,ótima qualidade.
3,Bom.,bom.
4,Atendeu minhas expectativas.,atendeu minhas expectativas.
5,Ótimo recomendo e já usei a fragrância é bem f...,ótimo recomendo e já usei a fragrância é bem f...
6,"Dentro do esperado, recomendo.","dentro do esperado, recomendo."
7,Amei 😍.,amei .


In [18]:
# ============================================================
# Resumo comparativo antes x depois da limpeza
# ============================================================

print("=" * 55)
print("RELATÓRIO DE LIMPEZA — ANTES x DEPOIS")
print("=" * 55)

print(f"\n📦 VOLUME")
print(f"  Registros originais:  {total_antes:,}")
print(f"  Registros após limpeza: {len(df):,}")
print(f"  Removidos (vazios):   {total_antes - len(df):,}")

print(f"\n🏷️  FLAGS")
print(f"  Reviews curtos (< 3 palavras): {df['review_curto'].sum():,}")

print(f"\n📅 DATAS")
print(f"  Formato: datetime ✅")
print(f"  Datas inválidas: {df['date'].isna().sum():,}")
print(f"  Range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n📝 TEXTO (content)")
print(f"  Emojis removidos ✅")
print(f"  Lowercase aplicado ✅")
print(f"  Caracteres especiais removidos ✅")
print(f"  Média de palavras por review: {df['qtd_palavras'].mean():.1f}")

print(f"\n⭐ RATINGS")
print(df["rating"].value_counts().sort_index().to_string())

RELATÓRIO DE LIMPEZA — ANTES x DEPOIS

📦 VOLUME
  Registros originais:  206,952
  Registros após limpeza: 202,785
  Removidos (vazios):   4,167

🏷️  FLAGS
  Reviews curtos (< 3 palavras): 54,545

📅 DATAS
  Formato: datetime ✅
  Datas inválidas: 0
  Range: 2015-11-01 → 2025-02-19

📝 TEXTO (content)
  Emojis removidos ✅
  Lowercase aplicado ✅
  Caracteres especiais removidos ✅
  Média de palavras por review: 9.6

⭐ RATINGS
rating
1      7196
2      2777
3      5690
4     12571
5    174551


In [20]:
# ============================================================
# Salva o dataset limpo em CSV para as próximas etapas
# ============================================================

caminho_saida = PROCESSED_PATH / "reviews_limpos.csv"

df.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"✅ Dataset limpo exportado com sucesso!")
print(f"📁 Caminho: {caminho_saida}")
print(f"📊 Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"\n📋 Colunas do dataset limpo:")
for col in df.columns:
    print(f"   - {col}")
df.head()

✅ Dataset limpo exportado com sucesso!
📁 Caminho: D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\processed\reviews_limpos.csv
📊 Shape final: 202,785 linhas × 7 colunas

📋 Colunas do dataset limpo:
   - date
   - rating
   - content
   - product_url
   - qtd_palavras
   - review_curto
   - content_original


,date,rating,content,product_url,qtd_palavras,review_curto,content_original
0,2023-09-09,5,top.,https://produto.mercadolivre.com.br/MLB-314957...,1,True,Top.
1,2024-08-19,5,"produto bom, cumpre o que promete.",https://produto.mercadolivre.com.br/MLB-314957...,6,False,"Produto bom, cumpre o que promete."
2,2025-02-15,5,ótima qualidade.,https://produto.mercadolivre.com.br/MLB-314957...,2,True,Ótima qualidade.
3,2025-02-11,4,bom.,https://produto.mercadolivre.com.br/MLB-314957...,1,True,Bom.
4,2025-01-10,5,atendeu minhas expectativas.,https://produto.mercadolivre.com.br/MLB-314957...,3,False,Atendeu minhas expectativas.
